## Bonus: PyTorch Impelmentation of GloVe Algorithm

Implementing GloVe in PyTorch is a bit different from Word2Vec. Instead of iterating through a sliding window during training, we treat it as a **Weighted Least Squares Regression** problem. We assume we have already pre-calculated a "Co-occurrence Matrix" $X$.

Here is a clean, modular implementation.

### 1. The GloVe Model Architecture
The model consists of two embedding layers ($E$ and $\theta$) and two bias vectors. As wedescribed above, these are symmetric.

```python
import torch
import torch.nn as nn
import torch.optim as optim

class GloVeModel(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super(GloVeModel, self).__init__()
        # Symmetric embeddings
        self.in_embed = nn.Embedding(vocab_size, embed_size)
        self.out_embed = nn.Embedding(vocab_size, embed_size)
        
        # Biases for each word (crucial for the GloVe objective)
        self.in_bias = nn.Embedding(vocab_size, 1)
        self.out_bias = nn.Embedding(vocab_size, 1)
        
        # Initialize weights
        nn.init.uniform_(self.in_embed.weight, -0.1, 0.1)
        nn.init.uniform_(self.out_embed.weight, -0.1, 0.1)
        nn.init.zeros_(self.in_bias.weight)
        nn.init.zeros_(self.out_bias.weight)

    def forward(self, i_indices, j_indices):
        # i = context word, j = target word
        w_i = self.in_embed(i_indices)   # [batch, embed_size]
        w_j = self.out_embed(j_indices)  # [batch, embed_size]
        
        b_i = self.in_bias(i_indices).squeeze() # [batch]
        b_j = self.out_bias(j_indices).squeeze() # [batch]
        
        # Dot product: \theta_j^T * e_i
        dot_product = torch.sum(w_i * w_j, dim=1)
        
        # Prediction: \theta_j^T * e_i + b_i + b_j
        return dot_product + b_i + b_j
```

### 2. The Weighted Loss Function
We need to implement the weighting function $f(X_{ij})$ and the squared error against $\log(X_{ij})$.

```python
class GloVeLoss(nn.Module):
    def __init__(self, x_max=100, alpha=0.75):
        super(GloVeLoss, self).__init__()
        self.x_max = x_max
        self.alpha = alpha

    def forward(self, predictions, counts):
        # 1. Calculate f(X_ij) weights
        # weight = (x/x_max)^alpha if x < x_max else 1
        weights = torch.pow(counts / self.x_max, self.alpha)
        weights = torch.where(counts < self.x_max, weights, torch.ones_like(weights))
        
        # 2. Calculate squared error: (pred - log(counts))^2
        # Note: counts should be >= 1 here (handled by Dataset)
        loss = weights * torch.pow(predictions - torch.log(counts), 2)
        
        return torch.mean(loss)
```

### 3. The Dataset Logic
Unlike the Skip-gram dataset, this one returns the **pre-calculated count** for a specific pair.

```python
from torch.utils.data import Dataset, DataLoader

class GloVeDataset(Dataset):
    def __init__(self, co-occurrence_matrix):
        """
        co-occurrence_matrix: A dictionary or sparse matrix 
        where keys are (i, j) and values are counts X_ij.
        """
        # We only store pairs where count > 0
        self.indices = list(co-occurrence_matrix.keys())
        self.counts = list(co-occurrence_matrix.values())

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i, j = self.indices[idx]
        count = self.counts[idx]
        return torch.tensor(i), torch.tensor(j), torch.tensor(count, dtype=torch.float)
```

### 4. Training and Weight Averaging
Finally, we train and—as per the "symmetric roles" discussed—average the weights.

```python
# Initialization
model = GloVeModel(vocab_size=10000, embed_size=300)
criterion = GloVeLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
for epoch in range(20):
    for i, j, counts in dataloader:
        optimizer.zero_grad()
        preds = model(i, j)
        loss = criterion(preds, counts)
        loss.backward()
        optimizer.step()

# --- Post-Training: Symmetric Averaging ---
# As Pennington et al. suggested, averaging in_embed and out_embed 
# often results in more robust vectors.
final_embeddings = (model.in_embed.weight + model.out_embed.weight) / 2
```

Notice that the training loop for GloVe is much "cleaner" than Word2Vec. We aren't sampling negatives on the fly; the "negatives" are implicitly handled because pairs that never occur ($X_{ij}=0$) are excluded from the summation by the weighting function. This makes GloVe exceptionally fast if you have a high-performance way to build the co-occurrence matrix $X$ upfront.